# 🔥 Forge-v1 — a corrida oficial (Degrau 2, sub-issue #6)

Notebook da **corrida forge-v1** da forja de políticas-código (épica #1), protocolo **pré-registrado** em `docs/FORGE-V1-LOG.md`:

- **O loop:** 300 iterações (seed 42) — seleção de pai em 5 ilhas → **mutação por LLM** → **guarda AST anti-trapaça** → **avaliador stage1** (8 episódios, campo F0, seeds de TREINO) → **stage2** (64 episódios de validação, F0+F1) quando S1 ≥ 0,45 ou S1 é recorde. A iteração 0 é a semente (geração 0 = v0), avaliada nos dois estágios.
- **O mutador:** ensemble Gemini via endpoint OpenAI-compatible — `gemini-2.5-flash-lite` no volume (peso 0,8) e `gemini-2.5-pro` nos saltos (peso 0,2) — com **fallback Groq** (`openai/gpt-oss-120b`, depois `qwen/qwen3-32b`) se a Gemini falhar. Diff SEARCH/REPLACE restrito ao EVOLVE-BLOCK; o prompt carrega as métricas do pai e os artefatos do avaliador (resumo bloco a bloco + destaques da Crônica). Chaves **só** via Kaggle Secrets (`GEMINI_API_KEY` / `GROQ_API_KEY`) — a célula-sonda abaixo imprime apenas presente/ausente, nunca valores.
- **Bloqueio pré-registrado:** sem NENHUM dos dois secrets, o notebook imprime **FORGE-V1: BLOQUEADO — secrets ausentes** e termina limpo (ação do fundador: cadastrar os secrets; não é erro).
- **Avaliação cega (ao fim, fora do loop):** campeão da corrida × campeão do torneio (`evo:6a94b587701d`) × semente, todos como focal nos **64 blocos de TESTE** (32 seeds 9901–9932 × (F0, F2 com choques)) — a primeira abertura das seeds travadas, com `final=True`. ΔS pareado + IC95 por bootstrap contra cada baseline.
- **Veredito (barras do torneio):** ΔS(campeão − campeão do torneio) ≥ 0,08 com IC95_lo > 0, sem colapso em F2/choques e ΔS vs semente > 0 ⇒ **FORGE-V1: PROVA**; ΔS > 0 sem a barra ⇒ **INCONCLUSIVO**; ΔS ≤ 0 ⇒ **REPROVADO/AJUSTAR**. Relatório: `final_evaluation.json` + `top5_diffs.md` (os 5 passos de maior ganho da linhagem do campeão).

Regra do fundador: experimento roda AQUI, no Kaggle — nunca na VM de desenvolvimento.


In [ ]:
# 1) Fonte do pacote: dataset privado do bundle (mundo-autonomo-src12).
#    LOADER RECURSIVO: o Kaggle expoe os datasets em
#    /kaggle/input/datasets/<dono>/<slug>/ com a arvore ja extraida;
#    a busca aceita qualquer 'src/mundo/__init__.py' sob /kaggle/input,
#    com fallback para o zip mundo-src.zip. O bundle leva src/ +
#    forge/ + pyproject.toml -- a raiz do bundle entra no sys.path
#    porque a forja vive em forge/, fora do pacote.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    bundle_root = os.path.dirname(os.path.dirname(
        os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    bundle_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset pascalcoimbra/mundo-autonomo-src12 '
                     'a este kernel.')
print('bundle em', bundle_root)
sys.path.insert(0, os.path.join(bundle_root, 'src'))
sys.path.insert(0, bundle_root)
import mundo
print('mundo', mundo.__version__, 'importado de', bundle_root)
for rel in ('forge/run.py', 'forge/llm_mutator.py', 'forge/evaluator.py',
            'forge/seed_policy.py', 'forge/mutator.py',
            'forge/tournament_champion.json', 'src/mundo/program.py'):
    assert os.path.exists(os.path.join(bundle_root, rel)), \
        f'{rel} ausente no bundle'
print('forge/ completa no bundle (driver + mutador LLM + campeao do torneio)')


In [ ]:
# 2) SONDA DE SECRETS: imprime APENAS presente/ausente (nunca o valor).
#    As chaves vao para variaveis de ambiente, de onde o mutador le.
import os

def _probe(nome):
    valor = os.environ.get(nome)
    if not valor:
        try:
            from kaggle_secrets import UserSecretsClient
            valor = UserSecretsClient().get_secret(nome)
        except Exception:
            valor = None
    if valor:
        os.environ[nome] = valor
    print(f'{nome}:', 'presente' if valor else 'ausente')
    return bool(valor)

tem_gemini = _probe('GEMINI_API_KEY')
tem_groq = _probe('GROQ_API_KEY')
BLOQUEADO = not (tem_gemini or tem_groq)
if BLOQUEADO:
    print('FORGE-V1: BLOQUEADO \u2014 secrets ausentes')
    print('acao do fundador: cadastrar GEMINI_API_KEY e/ou GROQ_API_KEY '
          'como Secrets deste kernel e rodar de novo.')


In [ ]:
# 3) A corrida: 300 iteracoes do loop [pai -> mutacao LLM -> guarda
#    -> stage1 -> stage2] + avaliacao cega nas seeds de teste.
import importlib.util, json, os, time, traceback

summary = None
run_error = None
if BLOQUEADO:
    print('corrida nao iniciada: secrets ausentes (ver celula 2)')
else:
    spec = importlib.util.spec_from_file_location(
        'forge_run', os.path.join(bundle_root, 'forge', 'run.py'))
    forge_run = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(forge_run)
    t0 = time.time()
    try:
        summary = forge_run.run_forge(
            iterations=300, seed=42, out_dir='/kaggle/working/forge-v1',
            mutator_kind='llm', final=True, final_processes=4)
    except Exception:
        run_error = traceback.format_exc()
        print('a corrida quebrou:')
        print(run_error)
    print(f'celula da corrida concluida em {time.time() - t0:.0f}s')


In [ ]:
# 4) VEREDITO (regras pre-registradas no topo e em docs/FORGE-V1-LOG.md)
import json, os, shutil

print('=' * 64)
if BLOQUEADO:
    print('FORGE-V1: BLOQUEADO \u2014 secrets ausentes')
elif summary is None:
    print('FORGE-V1: FALHOU')
    print('causa:', (run_error or 'desconhecida')[-800:])
elif summary.get('verdict'):
    fe = summary['final_evaluation']
    print(summary['verdict'])
    print(f"campeao prog:{summary.get('champion_hash')}: "
          f"S1={summary['champion']['S1']} S2={summary['champion']['S2']}")
    print(f"semente prog:{summary['seed_result']['hash']}: "
          f"S1={summary['seed_result']['S1']} "
          f"S2={summary['seed_result']['S2']}")
    d = fe['delta_vs_tournament']
    print(f"\u0394S vs campeao do torneio ({fe['tournament_champion']}): "
          f"{d['mean']:+.4f} IC95 [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] "
          f"em {d['n_blocks']} blocos")
    print(f"\u0394S recorte F2: {fe['delta_vs_tournament_F2']['mean']:+.4f} | "
          f"recorte choques: {fe['delta_vs_tournament_shocks']['mean']:+.4f}")
    ds = fe['delta_vs_seed']
    print(f"\u0394S vs semente: {ds['mean']:+.4f} "
          f"IC95 [{ds['ci95'][0]:+.4f}, {ds['ci95'][1]:+.4f}]")
    print(f"S medio nos blocos de teste: {fe['mean_S']}")
    print(f"iteracoes: {summary.get('iterations_completed')}/300 | "
          f"contagens: {summary.get('counts')} | "
          f"mutador: {summary.get('mutator_stats')}")
    print(f"curva best_S1 (a cada 25): "
          f"{summary['s_curve_best_S1'][::25]}")
else:
    print('FORGE-V1: FALHOU')
    print('causa:', summary.get('fatal')
          or f"iteracoes completas: {summary.get('iterations_completed')}/300")
print('=' * 64)
if summary is not None:
    with open('/kaggle/working/forge-v1.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=1)
    for nome in ('champion.py', 'top5_diffs.md', 'final_evaluation.json',
                 'run_log.jsonl', 'archive.json'):
        _p = os.path.join('/kaggle/working/forge-v1', nome)
        if os.path.exists(_p):
            shutil.copy(_p, os.path.join('/kaggle/working', nome))
